<a href="https://colab.research.google.com/github/marikhakhishvili/Facial-Expression-Recognition-Challenge/blob/main/model4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install kaggle wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 22.2 MB/s eta 0:00:00


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
! mkdir ~/.kaggle

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [11]:
!cp /content/drive/MyDrive/cs231n/assignments/assignment4/kaggle.json ~/.kaggle/kaggle.json

In [12]:
! chmod 600 ~/.kaggle/kaggle.json

download competition data

In [13]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:01<00:00, 155MB/s]



In [14]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [15]:
import wandb
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Preprocessing

In [16]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

df = pd.read_csv('./train.csv')


def pixels_to_image_array(pixels_str):
    """
    Converts a space-separated pixel string into a normalized
    1 x 48 x 48 NumPy array suitable for PyTorch.
    """

    # Convert string to NumPy array of floats
    pixels = np.array(pixels_str.split(), dtype=np.float32)

    # Reshape into a 48x48 image
    image = pixels.reshape(48, 48)

    # Normalize pixel values from [0, 255] to [0, 1]
    image = image / 255.0

    # Add channel dimension: (48, 48) -> (1, 48, 48)
    image = np.expand_dims(image, axis=0)

    return image

# Apply preprocessing to every image
df["pixels"] = df["pixels"].apply(pixels_to_image_array)

# Check the shape of the first image
print(df["pixels"].iloc[0].shape)
# Expected output: (1, 48, 48)

(1, 48, 48)


In [17]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

X = np.stack(df["pixels"].values)   # shape: (N, 1, 48, 48)
y = df["emotion"].values

# First split: 70% for training, 30% for (validation + test)
X_train, X_val_test, y_train, y_val_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 50% of the 30% for validation (15% total), 50% of the 30% for test (15% total)
X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test, test_size=0.5, random_state=42, stratify=y_val_test
)

In [18]:
print("\nEmotion distribution in New Training Set:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nEmotion distribution in Validation Set:")
print(pd.Series(y_val).value_counts(normalize=True))


Emotion distribution in New Training Set:
3    0.251294
6    0.172920
4    0.168242
2    0.142715
0    0.139182
5    0.110470
1    0.015177
Name: proportion, dtype: float64

Emotion distribution in Validation Set:
3    0.251277
6    0.173014
4    0.168137
2    0.142592
0    0.139108
5    0.110543
1    0.015327
Name: proportion, dtype: float64


In [19]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05))
])

In [20]:
class FERDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]  # (1, 48, 48)
        label = self.labels[idx]


        # augmentation ONLY on training set
        if self.transform:
            # convert to PIL-like shape for torchvision transforms
            img = self.transform(img)

        return img, label # label is already a torch.long tensor

In [21]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

train_dataset = FERDataset(X_train, y_train, transform=train_transform)
val_dataset = FERDataset(X_val, y_val, transform=None)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model (same as model3)

In [22]:
import torch.nn as nn

class CNN_Model4(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [23]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / len(loader), correct / total

Training (with hyperparameters that worked well for model3)

In [24]:
wandb.init(
    project="fer2013-assignment",
    name="model4_augmented_earlystop",
    config={
        "model": "CNN_Model4",
        "lr": 5e-4,
        "batch_size": 64,
        "optimizer": "Adam",
        "weight_decay": 1e-4,
        "epochs": 25,
        "early_stopping_patience": 5,
        "augmentation": True,
    }
)

config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mkhak23 (mkhak23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model4().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.lr,
    weight_decay=config.weight_decay
)

In [26]:
best_val_loss = float("inf")
patience = config.early_stopping_patience
epochs_without_improvement = 0

In [27]:
best_val_loss = float("inf")
patience = 5
epochs_without_improvement = 0

for epoch in range(config.epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{config.epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        # Save the best model
        torch.save(model.state_dict(), "best_model4.pt")

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after epoch {epoch + 1}.")
            break

# Load the best-performing weights
model.load_state_dict(torch.load("best_model4.pt"))



Epoch 1/25
Train Loss: 1.7762, Train Acc: 0.2619
Val Loss:   1.6143, Val Acc:   0.3446
Epoch 2/25
Train Loss: 1.6083, Train Acc: 0.3583
Val Loss:   1.4949, Val Acc:   0.4275
Epoch 3/25
Train Loss: 1.5250, Train Acc: 0.3985
Val Loss:   1.4209, Val Acc:   0.4340
Epoch 4/25
Train Loss: 1.4882, Train Acc: 0.4125
Val Loss:   1.3996, Val Acc:   0.4533
Epoch 5/25
Train Loss: 1.4525, Train Acc: 0.4321
Val Loss:   1.4024, Val Acc:   0.4566
Epoch 6/25
Train Loss: 1.4254, Train Acc: 0.4421
Val Loss:   1.4050, Val Acc:   0.4680
Epoch 7/25
Train Loss: 1.3990, Train Acc: 0.4551
Val Loss:   1.3765, Val Acc:   0.4712
Epoch 8/25
Train Loss: 1.3938, Train Acc: 0.4519
Val Loss:   1.3980, Val Acc:   0.4754
Epoch 9/25
Train Loss: 1.3682, Train Acc: 0.4687
Val Loss:   1.3196, Val Acc:   0.4912
Epoch 10/25
Train Loss: 1.3578, Train Acc: 0.4737
Val Loss:   1.3289, Val Acc:   0.5028
Epoch 11/25
Train Loss: 1.3463, Train Acc: 0.4759
Val Loss:   1.3430, Val Acc:   0.5039
Epoch 12/25
Train Loss: 1.3365, Train Acc

<All keys matched successfully>

## Model Evaluation on Test Set

In [28]:
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

test_dataset = FERDataset(X_test, y_test, transform=None)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

test_loss, test_acc = evaluate(model, test_loader)

print(f"\nTest Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

wandb.log({
    "test_loss": test_loss,
    "test_acc": test_acc,
})
wandb.finish()



Test Loss: 1.1761, Test Acc: 0.5821


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_acc,▁
test_loss,▁
train_acc,▁▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇███████
train_loss,█▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁
val_acc,▁▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██
val_loss,█▆▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁
epoch,25
test_acc,0.58208
test_loss,1.17611
train_acc,0.52563
